In [3]:
import pandas as pd
import re
from pathlib import Path

# Folder chứa 2 file .xlsb gốc
folder_path = Path("/Users/locdang/Desktop/Power BI/Xu Ly chat thai 2023-2024/Wastes Received")

# Folder xuất file merge mới
output_folder = Path("/Users/locdang/Desktop/Power BI/Xu Ly chat thai 2023-2024/Data_clean")
output_folder.mkdir(parents=True, exist_ok=True)

# Tự tìm file .xlsb
excel_files = list(folder_path.glob("*Waste Data Interrogator - Wastes Received*Version 2*.xlsb"))

print("Các file .xlsb tìm được:")
for f in excel_files:
    print("-", f.name)

if not excel_files:
    raise FileNotFoundError("Không tìm thấy file .xlsb nào trong folder Wastes Received.")

dataframes = []

for file_path in excel_files:
    file_name = file_path.name

    # Lấy năm từ tên file
    year_match = re.search(r"\b(20\d{2})\b", file_name)
    if not year_match:
        print(f"Bỏ qua file vì không tìm thấy năm: {file_name}")
        continue

    year = int(year_match.group(1))

    # Tên sheet dự kiến
    expected_sheet = f"{year} Waste Received"

    # Mở file .xlsb
    excel_file = pd.ExcelFile(file_path, engine="pyxlsb")

    print(f"\nFile: {file_name}")
    print("Sheets:", excel_file.sheet_names)

    # Chọn đúng sheet
    if expected_sheet in excel_file.sheet_names:
        sheet_name = expected_sheet
    else:
        possible_sheets = [
            s for s in excel_file.sheet_names
            if "waste received" in s.lower()
        ]

        if not possible_sheets:
            raise ValueError(f"Không tìm thấy sheet Waste Received trong file: {file_name}")

        sheet_name = possible_sheets[0]

    print(f"Đang đọc sheet: {sheet_name}")

    # Đọc data
    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        engine="pyxlsb"
    )

    # Thêm cột year
    df["year"] = year

    dataframes.append(df)

if not dataframes:
    raise ValueError("Không có dữ liệu nào để merge.")

# Merge data
merged_df = pd.concat(dataframes, ignore_index=True)

# Xuất file mới vào Data_clean
output_file = output_folder / "Merged Waste Data Received 2023-2024.xlsx"
merged_df.to_excel(output_file, index=False, engine="openpyxl")

print("\nMerge xong!")
print(f"File xuất ra: {output_file}")
print(f"Tổng số dòng: {len(merged_df)}")
print(f"Tổng số cột: {len(merged_df.columns)}")

Các file .xlsb tìm được:
- 2023 Waste Data Interrogator - Wastes Received (Excel) - Version 2.xlsb
- 2024 Waste Data Interrogator - Wastes Received (Excel) - Version 2.xlsb

File: 2023 Waste Data Interrogator - Wastes Received (Excel) - Version 2.xlsb
Sheets: ['Information', 'Glossary', '2023 Waste Received', 'Interrogator - Waste Received', 'Active Sites', 'Configurable Reports', 'One Click Summaries', 'Waste Data Links', 'European Waste Catalogue']
Đang đọc sheet: 2023 Waste Received

File: 2024 Waste Data Interrogator - Wastes Received (Excel) - Version 2.xlsb
Sheets: ['Information', 'Glossary', '2024 Waste Received', 'Interrogator - Waste Received', 'Active Sites', 'Configurable Reports', 'One Click Summaries', 'Waste Data Links', 'European Waste Catalogue', 'List of Changes']
Đang đọc sheet: 2024 Waste Received

Merge xong!
File xuất ra: /Users/locdang/Desktop/Power BI/Xu Ly chat thai 2023-2024/Data_clean/Merged Waste Data Received 2023-2024.xlsx
Tổng số dòng: 683534
Tổng số cột: 